# SQL

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from IPython.display import display
from sql.queries import QUERIES
from utils import db_connection as db

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)


def show(key, rows=15):
    """Print the question title and SQL, then show the result table."""
    q = QUERIES[key]
    print(f"{key.upper()} [{q['level']}] {q['title']}")
    print("-" * 70)
    print(q["sql"], "\n")
    df = db.run_query(q["sql"])
    print(f"{len(df)} rows")
    display(df.head(rows))
    return df

In [2]:
df = show("q17")

Q17 [Advanced] Does winning the toss help?
----------------------------------------------------------------------
WITH decided AS (
    SELECT toss_decision, toss_winner_id, winner_id
    FROM matches
    WHERE toss_winner_id IS NOT NULL
      AND toss_decision  IS NOT NULL
      AND winner_id      IS NOT NULL
),
by_decision AS (
    SELECT toss_decision AS decision,
           COUNT(*) AS matches,
           SUM(CASE WHEN toss_winner_id = winner_id THEN 1 ELSE 0 END) AS toss_winner_won
    FROM decided
    GROUP BY toss_decision
    UNION ALL
    SELECT 'overall', COUNT(*),
           SUM(CASE WHEN toss_winner_id = winner_id THEN 1 ELSE 0 END)
    FROM decided
)
SELECT decision, matches, toss_winner_won,
       ROUND(100.0 * toss_winner_won / matches, 2) AS win_pct
FROM by_decision
ORDER BY CASE decision WHEN 'bat' THEN 1 WHEN 'bowl' THEN 2 ELSE 3 END; 

3 rows


,decision,matches,toss_winner_won,win_pct
0,bat,219,123,56.16
1,bowl,213,108,50.70
2,overall,432,231,53.47


In [3]:
df = show("q18")

Q18 [Advanced] Most economical limited-overs bowlers
----------------------------------------------------------------------
-- overs are stored as 3.4 = 3 overs 4 balls, so convert to true overs first
WITH spells AS (
    SELECT bp.player_id, bp.match_id, bp.runs_conceded, bp.wickets,
           CAST(bp.overs AS INTEGER)
             + (bp.overs - CAST(bp.overs AS INTEGER)) * 10.0 / 6 AS true_overs
    FROM bowling_performances bp
    JOIN matches m ON m.match_id = bp.match_id
    WHERE m.match_format IN ('ODI', 'T20I', 'T20')
)
SELECT p.full_name,
       COUNT(DISTINCT s.match_id)                      AS matches_bowled,
       ROUND(SUM(s.true_overs), 1)                     AS total_overs,
       SUM(s.wickets)                                  AS total_wickets,
       ROUND(SUM(s.runs_conceded) / SUM(s.true_overs), 2) AS economy_rate
FROM spells s
JOIN players p ON p.player_id = s.player_id
GROUP BY s.player_id, p.full_name
HAVING COUNT(DISTINCT s.match_id) >= 10
   AND SUM(s.true_ove

,full_name,matches_bowled,total_overs,total_wickets,economy_rate
0,Archie Calloway,73,508.0,112,5.63
1,Tane McRae,68,452.0,109,5.64
2,Hasitha Peiris,68,440.0,100,5.65
3,Archie Whitmore,73,520.0,137,5.71
4,Dinesh Gunaratne,71,482.0,123,5.76
5,Faizan Abbasi,71,494.0,107,5.77
6,Pasindu Peiris,74,506.0,115,5.77
7,Dinesh Rajapaksa,73,472.0,130,5.77
8,Archie Pemberton,64,430.0,98,5.83
9,Theo Pemberton,70,490.0,119,5.87


In [4]:
df = show("q19")

Q19 [Advanced] Batting consistency since 2022
----------------------------------------------------------------------
-- at least 5 innings are required so the standard deviation means something
SELECT p.full_name,
       COUNT(*)                  AS innings,
       ROUND(AVG(b.runs), 2)     AS avg_runs,
       ROUND(STDDEV(b.runs), 2)  AS std_dev_runs
FROM batting_performances b
JOIN matches m ON m.match_id = b.match_id
JOIN players p ON p.player_id = b.player_id
WHERE m.match_date >= '2022-01-01'
  AND b.balls_faced >= 10
GROUP BY b.player_id, p.full_name
HAVING COUNT(*) >= 5
ORDER BY std_dev_runs ASC; 

86 rows


,full_name,innings,avg_runs,std_dev_runs
0,Rashid Springer,10,8.20,2.35
1,Aditya Nair,10,9.50,2.92
2,Dinesh Gunaratne,6,8.00,2.97
3,Rhys McRae,5,9.40,3.44
4,Dinesh Rajapaksa,19,10.58,3.98
5,Zeeshan Qureshi,14,8.71,4.56
6,Harvey Calloway,22,11.59,4.67
7,Vikram Patel,23,9.83,4.69
8,Cameron Fraser,10,11.00,5.06
9,Hamza Mirza,21,10.24,6.66


In [5]:
df = show("q20")

Q20 [Advanced] Matches and batting average by format (20+ matches)
----------------------------------------------------------------------
SELECT p.full_name,
       SUM(CASE WHEN c.format = 'Test' THEN c.matches END) AS test_matches,
       SUM(CASE WHEN c.format = 'ODI'  THEN c.matches END) AS odi_matches,
       SUM(CASE WHEN c.format = 'T20I' THEN c.matches END) AS t20_matches,
       MAX(CASE WHEN c.format = 'Test' THEN c.batting_avg END) AS test_avg,
       MAX(CASE WHEN c.format = 'ODI'  THEN c.batting_avg END) AS odi_avg,
       MAX(CASE WHEN c.format = 'T20I' THEN c.batting_avg END) AS t20_avg
FROM player_career_stats c
JOIN players p ON p.player_id = c.player_id
GROUP BY c.player_id, p.full_name
HAVING SUM(c.matches) >= 20
ORDER BY SUM(c.matches) DESC; 

119 rows


,full_name,test_matches,odi_matches,t20_matches,test_avg,odi_avg,t20_avg
0,Jason Maynard,105.0,196.0,113.0,39.18,30.44,26.72
1,Dinesh Abeywickrama,58.0,249.0,103.0,51.88,37.16,30.02
2,Usman Abbasi,87.0,240.0,74.0,48.84,53.81,27.03
3,Archie Hartley,55.0,210.0,119.0,17.30,12.69,8.15
4,Lwazi Mokoena,77.0,206.0,87.0,15.08,14.29,10.71
5,Dewald Naidoo,51.0,233.0,80.0,8.73,17.87,9.89
6,Rohan Patel,50.0,206.0,90.0,38.32,32.66,25.07
7,Vikram Mehta,107.0,174.0,61.0,37.30,47.26,28.67
8,Faizan Qureshi,85.0,173.0,83.0,29.30,40.67,22.52
9,Freddie Hartley,62.0,223.0,54.0,40.23,46.36,28.73


In [6]:
df = show("q21")

Q21 [Advanced] Weighted performance ranking (top 10 per format)
----------------------------------------------------------------------
WITH scored AS (
    SELECT c.player_id, c.format,
           c.runs * 0.01 + COALESCE(c.batting_avg, 0) * 0.5
             + COALESCE(c.strike_rate, 0) * 0.3                         AS batting_points,
           CASE WHEN c.wickets > 0
                THEN c.wickets * 2
                     + (50 - COALESCE(c.bowling_avg, 50)) * 0.5
                     + (6  - COALESCE(c.economy_rate, 6)) * 2
                ELSE 0 END                                              AS bowling_points,
           c.catches * 3 + c.stumpings * 5                              AS fielding_points
    FROM player_career_stats c
),
ranked AS (
    SELECT s.*,
           batting_points + bowling_points + fielding_points AS total_points,
           RANK() OVER (PARTITION BY format
                        ORDER BY batting_points + bowling_points + fielding_points DESC) AS rank_in_f

,format,rank,full_name,batting_pts,bowling_pts,fielding_pts,total_pts
0,Test,1,Jacob Henare,53.0,723.8,180.0,956.8
1,Test,2,Aditya Nair,37.9,651.6,105.0,794.5
2,Test,3,Cameron Fraser,38.1,641.7,78.0,757.8
3,Test,4,Vikram Patel,32.8,459.0,228.0,719.8
4,Test,5,Rashid Springer,52.9,582.3,75.0,710.1
5,Test,6,Pasindu Rajapaksa,37.1,483.6,189.0,709.7
6,Test,7,Tyrone Burrowes,98.0,318.7,291.0,707.7
7,Test,8,Hasitha Peiris,109.3,283.3,297.0,689.6
8,Test,9,Harvey Calloway,28.7,408.4,186.0,623.1
9,Test,10,Jason Maynard,101.0,245.8,273.0,619.8


In [7]:
df = show("q22")

Q22 [Advanced] Head-to-head record (pairs with 5+ matches in the last 3 years)
----------------------------------------------------------------------
WITH pair_matches AS (
    SELECT MIN(team1_id, team2_id) AS team_a,
           MAX(team1_id, team2_id) AS team_b,
           winner_id, victory_type, victory_margin
    FROM matches
    WHERE state = 'Complete'
      AND match_date >= date('now', '-3 year')
)
SELECT ta.team_name AS team_a,
       tb.team_name AS team_b,
       COUNT(*) AS total_matches,
       SUM(CASE WHEN pm.winner_id = pm.team_a THEN 1 ELSE 0 END) AS wins_a,
       SUM(CASE WHEN pm.winner_id = pm.team_b THEN 1 ELSE 0 END) AS wins_b,
       ROUND(AVG(CASE WHEN pm.winner_id = pm.team_a AND pm.victory_type = 'runs'    THEN pm.victory_margin END), 1) AS a_avg_runs_margin,
       ROUND(AVG(CASE WHEN pm.winner_id = pm.team_a AND pm.victory_type = 'wickets' THEN pm.victory_margin END), 1) AS a_avg_wkts_margin,
       ROUND(AVG(CASE WHEN pm.winner_id = pm.team_b AND pm.victor

,team_a,team_b,total_matches,wins_a,wins_b,a_avg_runs_margin,a_avg_wkts_margin,b_avg_runs_margin,b_avg_wkts_margin,win_pct_a,win_pct_b
0,India,West Indies,10,6,3,81.0,2.5,81.7,NaN,60.0,30.0
1,Pakistan,Sri Lanka,9,4,5,82.0,1.7,49.3,5.0,44.4,55.6
2,Australia,South Africa,8,6,2,136.0,2.0,93.5,NaN,75.0,25.0
3,Pakistan,West Indies,8,5,3,112.7,1.5,66.0,3.0,62.5,37.5
4,Sri Lanka,New Zealand,8,3,5,29.0,1.0,214.0,1.0,37.5,62.5
5,Australia,Sri Lanka,7,5,2,153.0,1.0,276.0,4.0,71.4,28.6
6,Australia,West Indies,7,5,2,36.0,1.5,8.0,2.0,71.4,28.6
7,Australia,England,7,2,5,NaN,1.0,96.0,2.0,28.6,71.4
8,Australia,New Zealand,7,5,2,157.8,1.0,87.0,1.0,71.4,28.6
9,England,New Zealand,7,5,2,283.0,3.0,124.0,1.0,71.4,28.6


In [8]:
df = show("q22b")

Q22B [Advanced] Head-to-head: batting first vs bowling first by venue
----------------------------------------------------------------------
WITH oriented AS (
    SELECT m.match_id, m.venue_id, m.winner_id,
           MIN(m.team1_id, m.team2_id) AS team_a,
           MAX(m.team1_id, m.team2_id) AS team_b,
           CASE WHEN m.toss_decision = 'bat' THEN m.toss_winner_id
                WHEN m.toss_winner_id = m.team1_id THEN m.team2_id
                ELSE m.team1_id END AS bat_first_id
    FROM matches m
    WHERE m.state = 'Complete'
      AND m.match_date >= date('now', '-3 year')
      AND m.toss_winner_id IS NOT NULL
      AND m.toss_decision  IS NOT NULL
      AND m.winner_id      IS NOT NULL
),
eligible AS (
    SELECT MIN(team1_id, team2_id) AS team_a,
           MAX(team1_id, team2_id) AS team_b
    FROM matches
    WHERE state = 'Complete' AND match_date >= date('now', '-3 year')
    GROUP BY 1, 2
    HAVING COUNT(*) >= 5
)
SELECT ta.team_name AS team_a, tb.team_name AS tea

,team_a,team_b,venue_name,matches,won_batting_first,won_bowling_first
0,Australia,England,Lord's,3,1,2
1,Australia,England,Eden Park,2,1,1
2,Australia,England,Sydney Cricket Ground,2,1,1
3,Australia,New Zealand,Galle International Stadium,2,1,1
4,Australia,New Zealand,Eden Park,2,2,0
5,Australia,New Zealand,Melbourne Cricket Ground,2,1,1
6,Australia,New Zealand,Newlands,1,1,0
7,Australia,South Africa,Newlands,3,2,1
8,Australia,South Africa,Gaddafi Stadium,1,0,1
9,Australia,South Africa,Eden Park,1,1,0


In [9]:
df = show("q23")

Q23 [Advanced] Recent form: last 10 batting performances
----------------------------------------------------------------------
-- Form rules (documented so they can be tuned):
--   Excellent: last-5 average >= 40 AND at least 3 scores above 50 in last 10
--   Good: last-5 average >= 28    Average: >= 18    otherwise Poor
WITH ranked AS (
    SELECT b.player_id, b.runs, b.strike_rate,
           ROW_NUMBER() OVER (PARTITION BY b.player_id
                              ORDER BY m.match_date DESC, m.match_id DESC, b.innings_no DESC) AS rn
    FROM batting_performances b
    JOIN matches m ON m.match_id = b.match_id
    WHERE b.strike_rate IS NOT NULL AND m.match_date IS NOT NULL
),
agg AS (
    SELECT player_id,
           AVG(CASE WHEN rn <= 5 THEN runs END)          AS avg_last5,
           AVG(runs)                                     AS avg_last10,
           AVG(CASE WHEN rn <= 5 THEN strike_rate END)   AS sr_last5,
           AVG(CASE WHEN rn >  5 THEN strike_rate END)   AS sr_prev

,full_name,avg_last5,avg_last10,strike_rate_trend,scores_over_50,std_dev_runs,variability,form
0,Hasitha Jayasekara,61.2,50.7,-19.0,5,40.7,0.80,Excellent Form
1,Kabir Reddy,60.8,62.6,15.8,6,40.6,0.65,Excellent Form
2,Kamran Abbasi,56.4,48.9,29.2,4,38.5,0.79,Excellent Form
3,Kamran Bukhari,51.6,40.1,22.5,2,24.1,0.60,Good Form
4,Freddie Hartley,47.4,56.1,11.8,6,33.4,0.60,Excellent Form
5,Freddie Whitmore,46.6,54.8,-1.9,4,41.7,0.76,Excellent Form
6,Harry Stanton,46.0,36.1,18.5,2,55.1,1.53,Good Form
7,Oshada Samarasinghe,45.4,46.4,-19.6,5,39.3,0.85,Excellent Form
8,Vikram Mehta,42.6,47.4,7.8,4,46.4,0.98,Excellent Form
9,Pasindu Gunaratne,37.6,29.1,-12.8,2,30.5,1.05,Good Form


In [10]:
df = show("q24")

Q24 [Advanced] Best batting partnerships (5+ together)
----------------------------------------------------------------------
WITH pairs AS (
    SELECT MIN(a.player_id, b.player_id) AS p1,
           MAX(a.player_id, b.player_id) AS p2,
           a.runs + b.runs AS partnership_runs
    FROM batting_performances a
    JOIN batting_performances b
      ON  b.match_id = a.match_id
      AND b.innings_no = a.innings_no
      AND b.batting_position = a.batting_position + 1
),
agg AS (
    SELECT p1, p2,
           COUNT(*) AS partnerships,
           AVG(partnership_runs) AS avg_runs,
           SUM(CASE WHEN partnership_runs > 50 THEN 1 ELSE 0 END) AS over_50,
           MAX(partnership_runs) AS highest
    FROM pairs
    GROUP BY p1, p2
    HAVING COUNT(*) >= 5
)
SELECT RANK() OVER (ORDER BY 100.0 * over_50 / partnerships DESC, avg_runs DESC) AS rank,
       pa.full_name AS player_1, pb.full_name AS player_2,
       partnerships,
       ROUND(avg_runs, 1) AS avg_partnership,
       over

,rank,player_1,player_2,partnerships,avg_partnership,partnerships_over_50,highest_partnership,success_rate_pct
0,1,Kabir Reddy,Kabir Patel,135,84.1,93,269,68.9
1,2,Liam Dawson,Liam Walker,138,76.3,90,275,65.2
2,3,Pieter Venter,Johan Pretorius,133,76.2,84,337,63.2
3,4,Kabir Patel,Vikram Mehta,135,83.0,82,345,60.7
4,5,Andre Burrowes,Marlon Maynard,126,68.3,74,196,58.7
5,6,Oliver Thornton,George Pemberton,131,63.7,76,178,58.0
6,7,Freddie Hartley,Oliver Thornton,131,69.3,75,216,57.3
7,8,Freddie Whitmore,Freddie Hartley,131,73.1,74,248,56.5
8,9,Salman Siddiqui,Kamran Abbasi,141,71.6,79,277,56.0
9,10,Kamran Abbasi,Kamran Bukhari,141,70.2,79,250,56.0


In [11]:
df = show("q25")

Q25 [Advanced] Career phase from quarterly performance (summary)
----------------------------------------------------------------------
-- Looks at each player's most recent 6-8 qualifying quarters (3+ matches each).
-- Newer half vs older half: +10% = ascending, -10% = declining, else stable.
WITH q AS (
    SELECT b.player_id,
           strftime('%Y', m.match_date) || '-Q'
             || ((CAST(strftime('%m', m.match_date) AS INTEGER) + 2) / 3) AS quarter,
           COUNT(DISTINCT b.match_id) AS matches,
           AVG(b.runs)        AS avg_runs,
           AVG(b.strike_rate) AS avg_sr
    FROM batting_performances b
    JOIN matches m ON m.match_id = b.match_id
    WHERE m.match_date IS NOT NULL
    GROUP BY b.player_id, quarter
    HAVING COUNT(DISTINCT b.match_id) >= 3
),
numbered AS (
    SELECT q.*,
           ROW_NUMBER() OVER (PARTITION BY player_id ORDER BY quarter DESC) AS rn_desc,
           COUNT(*)     OVER (PARTITION BY player_id) AS total_quarters
    FROM q
),
recen

,full_name,quarters_available,quarters_used,older_avg_runs,newer_avg_runs,older_avg_sr,newer_avg_sr,runs_change_pct,career_phase
0,Callan Cooper,17,8,3.4,7.3,79.3,90.0,118.9,Career Ascending
1,Freddie Hartley,17,8,22.0,43.0,86.1,83.5,95.5,Career Ascending
2,Oshada Samarasinghe,25,8,20.4,39.0,93.2,87.7,91.1,Career Ascending
3,Callan McRae,23,8,24.2,43.0,86.5,92.8,77.9,Career Ascending
4,Aditya Verma,15,8,5.2,9.0,69.9,77.0,74.2,Career Ascending
5,Tane McRae,14,8,3.9,6.5,73.6,56.9,68.5,Career Ascending
6,Andre Redman,14,8,2.9,4.9,45.3,80.8,66.7,Career Ascending
7,Hasitha Peiris,25,8,7.1,11.6,105.0,82.3,62.9,Career Ascending
8,Kamran Bukhari,23,8,31.5,51.1,93.5,103.7,62.3,Career Ascending
9,Harry Stanton,17,8,13.8,21.9,79.1,96.2,58.3,Career Ascending


In [12]:
df = show("q25b")

Q25B [Advanced] Quarter-by-quarter detail with change vs previous quarter
----------------------------------------------------------------------
WITH q AS (
    SELECT b.player_id,
           strftime('%Y', m.match_date) || '-Q'
             || ((CAST(strftime('%m', m.match_date) AS INTEGER) + 2) / 3) AS quarter,
           COUNT(DISTINCT b.match_id) AS matches,
           AVG(b.runs)        AS avg_runs,
           AVG(b.strike_rate) AS avg_sr
    FROM batting_performances b
    JOIN matches m ON m.match_id = b.match_id
    WHERE m.match_date IS NOT NULL
    GROUP BY b.player_id, quarter
    HAVING COUNT(DISTINCT b.match_id) >= 3
),
eligible AS (
    SELECT player_id FROM q GROUP BY player_id HAVING COUNT(*) >= 6
),
changes AS (
    SELECT q.*,
           LAG(avg_runs) OVER (PARTITION BY player_id ORDER BY quarter) AS prev_avg_runs
    FROM q
    WHERE player_id IN (SELECT player_id FROM eligible)
)
SELECT p.full_name, c.quarter, c.matches,
       ROUND(c.avg_runs, 1) AS avg_runs,
    

,full_name,quarter,matches,avg_runs,avg_strike_rate,change_pct,trend
0,Aditya Nair,2020-Q1,6,2.7,36.1,NaN,first quarter
1,Aditya Nair,2020-Q2,3,1.0,66.7,-62.5,declining
2,Aditya Nair,2020-Q4,3,2.0,61.5,100.0,improving
3,Aditya Nair,2021-Q2,3,3.8,54.7,87.5,improving
4,Aditya Nair,2021-Q3,3,2.7,133.3,-28.9,declining
5,Aditya Nair,2021-Q4,4,2.7,42.1,1.8,stable
6,Aditya Nair,2022-Q1,5,3.6,87.0,31.6,improving
7,Aditya Nair,2022-Q3,3,3.0,116.7,-16.0,declining
8,Aditya Nair,2023-Q1,5,2.5,78.2,-16.7,declining
9,Aditya Nair,2023-Q2,3,3.3,150.0,33.3,improving


# final cell

In [13]:
from pathlib import Path

lines = ["-- Cricbuzz LiveStats: 25 analytical queries (SQLite)\n"]
for key, q in QUERIES.items():
    lines.append(f"-- {key.upper()} [{q['level']}] {q['title']}\n{q['sql']}\n")
Path("../sql/queries.sql").write_text("\n".join(lines), encoding="utf-8")
print("Wrote sql/queries.sql")

Wrote sql/queries.sql
